# Query Decomposition RAG Pipeline
### PDF → Chunking → FAISS → Decompose → Sub-Retrieve → Sub-Answer → Synthesize

In [9]:
!pip install langchain langchain-community langchain-ollama langchain-text-splitters faiss-cpu pypdf -q

^C



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_ollama import OllamaEmbeddings, ChatOllama

c:\Users\shiva\.pyenv\pyenv-win\versions\3.12.10\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step 1: Load PDF

In [2]:
loader = PyPDFLoader("A._P._J._Abdul_Kalam.pdf")
pages = loader.load()

print(f"Total Pages: {len(pages)}")
print(f"Sample Content:\n{pages[0].page_content[:500]}")

Total Pages: 30
Sample Content:
A. P. J. Abdul Kalam
Official portrait, c. 2002
President of India
In office
25 July 2002 – 25 July 2007
Prime Minister Atal Bihari Vajpayee
Manmohan Singh
Vice President Krishan Kant
Bhairon Singh Shekhawat
Preceded by K. R. Narayanan
Succeeded by Pratibha Patil
Principal Scientific Adviser to the
Government of India
In office
November 1999 – November 2001
President K. R. Narayanan
Prime Minister Atal Bihari Vajpayee
Preceded by Office established
Succeeded by Rajagopala Chidambaram
Director Ge


## Step 2: Split into Chunks

In [3]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(pages)

print(f"Total Chunks: {len(chunks)}")

Total Chunks: 136


## Step 3: Create Embeddings & Store in FAISS

In [4]:
embeddings = OllamaEmbeddings(model="nomic-embed-text:latest")

vector_store = FAISS.from_documents(chunks, embeddings)
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

print(f"Vector Store Created with {vector_store.index.ntotal} vectors")

Vector Store Created with 136 vectors


## Step 4: Setup LLM

In [5]:
llm = ChatOllama(model="llama3.2:3b", temperature=0)

## Step 5: Decompose Query into Sub-Questions

In [6]:
query = "What were Abdul Kalam's contributions to India's space and missile programs, and how did they influence his presidency?"

decompose_prompt = f"""Break the following complex question into 2-4 simpler sub-questions.
Return ONLY the sub-questions, one per line, numbered.

Question: {query}

Sub-questions:"""

response = llm.invoke(decompose_prompt)
sub_questions = [q.strip().lstrip("0123456789.) ") for q in response.content.strip().split("\n") if q.strip()]

print("Original Query:", query)
print("\nSub-Questions:")
for i, sq in enumerate(sub_questions, 1):
    print(f"  {i}. {sq}")

Original Query: What were Abdul Kalam's contributions to India's space and missile programs, and how did they influence his presidency?

Sub-Questions:
  1. What were Abdul Kalam's key contributions to India's space program?
  2. How did Abdul Kalam's experience in the missile program influence his approach to national security during his presidency?
  3. What specific achievements did Abdul Kalam attribute to his time as the Chairman of the Space Commission?
  4. How did Abdul Kalam's leadership style and vision for India's space program impact his presidency?


## Step 6: Retrieve & Answer Each Sub-Question

In [7]:
sub_answers = []

for i, sq in enumerate(sub_questions, 1):
    # Retrieve relevant chunks for this sub-question
    docs = retriever.invoke(sq)
    context = "\n\n".join([doc.page_content for doc in docs])

    # Answer the sub-question
    sub_prompt = f"""Answer the question based only on the following context:

{context}

Question: {sq}
Answer:"""

    answer = llm.invoke(sub_prompt)
    sub_answers.append(answer.content)

    print(f"--- Sub-Q {i}: {sq}")
    print(f"    Answer: {answer.content}\n")

--- Sub-Q 1: What were Abdul Kalam's key contributions to India's space program?
    Answer: The text does not explicitly mention Abdul Kalam's key contributions to India's space program. However, it does mention that he:

* Was involved in the design of small hovercraft
* Worked on an expandable rocket project independently at DRDO in 1965
* Became the project director of India's first satellite launch vehicle (SLV) in 1969
* Successfully deployed the Rohini satellite in near-earth orbit in July 1980

These points suggest that Kalam made significant contributions to India's space program, particularly in the areas of rocketry and satellite technology.

--- Sub-Q 2: How did Abdul Kalam's experience in the missile program influence his approach to national security during his presidency?
    Answer: The text does not mention Abdul Kalam's presidency or his approach to national security during that time. It only mentions his experience in the missile program and his role in the developme

## Step 7: Synthesize Final Answer

In [8]:
# Combine sub-answers for synthesis
sub_qa_text = "\n".join([f"Q: {sq}\nA: {sa}" for sq, sa in zip(sub_questions, sub_answers)])

synthesis_prompt = f"""Based on the following sub-questions and their answers, provide a comprehensive answer to the original question.

Original Question: {query}

Sub-Questions and Answers:
{sub_qa_text}

Comprehensive Answer:"""

final_response = llm.invoke(synthesis_prompt)
print("Final Answer:", final_response.content)

Final Answer: Based on the sub-questions and their answers, Abdul Kalam's contributions to India's space and missile programs were significant, but his presidency is not explicitly mentioned in the text. However, it can be inferred that his experience in the missile program and his leadership style and vision for India's space program likely influenced his approach to national security during his presidency.

Kalam's contributions to India's space program were notable, particularly in the areas of rocketry and satellite technology. He was involved in the design of small hovercraft, worked on an expandable rocket project independently at DRDO in 1965, became the project director of India's first satellite launch vehicle (SLV) in 1969, and successfully deployed the Rohini satellite in near-earth orbit in July 1980. These achievements suggest that Kalam made significant contributions to India's space program.

Kalam's experience in the missile program also had an impact on his approach to

## Test Questions

1. How did Kalam's early life in Rameswaram shape his career in science and his views as president?
2. What roles did Kalam play in DRDO and ISRO, and what awards did he receive for his work?
3. What was Kalam's vision for India's development and how did he promote it after his presidency?